In [17]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

Libraries loaded.


In [18]:
df = pd.read_csv('../data/processed/merged/master.csv')

# timestamp_w is a datetime column — parse it so operations on it don't fail
df['timestamp_w'] = pd.to_datetime(df['timestamp_w'], errors='coerce')

print(f"master.csv loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print("\nColumn dtypes:")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

master.csv loaded: 239,919 rows × 19 cols

Column dtypes:
distance_km                 float64
travel_time_min             float64
num_lanes                     int64
num_signals                   int64
hour                        float64
day_of_week                   int64
avg_speed_kmph              float64
temperature_c               float64
visibility_km               float64
timestamp_w          datetime64[ns]
is_rush_hour                  int64
traffic_level_ord           float64
road_highway                  int64
road_residential              int64
road_urban                    int64
weather_clear                 int64
weather_fog                   int64
weather_rain                  int64
log_travel_time             float64
dtype: object

Null counts:
distance_km           1074
travel_time_min          0
num_lanes                0
num_signals              0
hour                     0
day_of_week              0
avg_speed_kmph           0
temperature_c            0
visibility_km 

In [19]:
EXPECTED_COLS = [
    'distance_km', 'travel_time_min', 'num_lanes', 'num_signals',
    'hour', 'day_of_week', 'avg_speed_kmph', 'temperature_c',
    'visibility_km', 'timestamp_w', 'is_rush_hour', 'traffic_level_ord',
    'road_highway', 'road_residential', 'road_urban',
    'weather_clear', 'weather_fog', 'weather_rain', 'log_travel_time'
]

missing = [c for c in EXPECTED_COLS if c not in df.columns]
extra   = [c for c in df.columns if c not in EXPECTED_COLS]

print(f"Expected cols missing from master: {missing if missing else 'None ✓'}")
print(f"Extra cols not in expected list:   {extra if extra else 'None ✓'}")

Expected cols missing from master: None ✓
Extra cols not in expected list:   None ✓


In [20]:
df = df.copy()

# --- road_type: decode from three binary columns ---
def decode_road_type(row):
    if row['road_highway'] == 1:
        return 'highway'
    elif row['road_residential'] == 1:
        return 'residential'
    elif row['road_urban'] == 1:
        return 'urban'
    else:
        return np.nan  # all-zero row → no road info available

df['road_type'] = df.apply(decode_road_type, axis=1)

# --- weather_type: decode from three binary columns ---
def decode_weather_type(row):
    if row['weather_clear'] == 1:
        return 'clear'
    elif row['weather_fog'] == 1:
        return 'fog'
    elif row['weather_rain'] == 1:
        return 'rain'
    else:
        return np.nan

df['weather_type'] = df.apply(decode_weather_type, axis=1)

# --- traffic_level: decode from ordinal column ---
traffic_ord_map = {0.0: 'low', 1.0: 'medium', 2.0: 'high'}
df['traffic_level'] = df['traffic_level_ord'].map(traffic_ord_map)

print("Decoded columns added:")
print(f"  road_type    — {df['road_type'].value_counts(dropna=False).to_dict()}")
print(f"  weather_type — {df['weather_type'].value_counts(dropna=False).to_dict()}")
print(f"  traffic_level— {df['traffic_level'].value_counts(dropna=False).to_dict()}")

Decoded columns added:
  road_type    — {'residential': 83603, 'urban': 72458, 'highway': 51306, nan: 32552}
  weather_type — {'clear': 172051, nan: 24717, 'rain': 21690, 'fog': 21461}
  traffic_level— {'low': 78387, 'medium': 70102, 'high': 58660, nan: 32770}


In [21]:
skewness = stats.skew(df['travel_time_min'].dropna())
print(f"Target skewness (travel_time_min): {skewness:.4f}")

USE_LOG_TRANSFORM = abs(skewness) > 1.0

if USE_LOG_TRANSFORM:
    print("→ |skewness| > 1.0: applying log1p transform. Using log_travel_time as target.")
else:
    print("→ |skewness| ≤ 1.0: no transform needed. Using travel_time_min as target.")

Target skewness (travel_time_min): 8.4659
→ |skewness| > 1.0: applying log1p transform. Using log_travel_time as target.


In [22]:
if USE_LOG_TRANSFORM:
    y = df['log_travel_time'].copy()
    print(f"Target: log_travel_time  |  shape: {y.shape}")
else:
    y = df['travel_time_min'].copy()
    print(f"Target: travel_time_min  |  shape: {y.shape}")

print(y.describe())

Target: log_travel_time  |  shape: (239919,)
count    239919.000000
mean          2.632876
std           0.912924
min           0.231112
25%           2.006871
50%           2.587012
75%           3.225653
max           6.397546
Name: log_travel_time, dtype: float64


In [23]:
# Columns that must never appear in X
COLS_TO_DROP = [
    'travel_time_min',   # raw target
    'log_travel_time',   # alt target
    'timestamp_w',       # raw datetime — no temporal features to extract (already done in cleaning NB)
    # Already-encoded columns that the Pipeline will re-encode from the raw strings
    'road_highway', 'road_residential', 'road_urban',
    'weather_clear', 'weather_fog', 'weather_rain',
    'traffic_level_ord',
]

X = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])

print(f"Feature matrix X: {X.shape[0]:,} rows × {X.shape[1]} cols")
print("\nColumns in X:")
print(X.columns.tolist())

Feature matrix X: 239,919 rows × 12 cols

Columns in X:
['distance_km', 'num_lanes', 'num_signals', 'hour', 'day_of_week', 'avg_speed_kmph', 'temperature_c', 'visibility_km', 'is_rush_hour', 'road_type', 'weather_type', 'traffic_level']


In [24]:
# Mirror of constants in src/train.py — single source of truth
NUM_FEATURES  = ['distance_km', 'num_lanes', 'num_signals', 'avg_speed_kmph',
                 'temperature_c', 'visibility_km', 'hour', 'day_of_week']
BOOL_FEATURES = ['is_rush_hour', 'is_weekend']
ORD_FEATURES  = ['traffic_level']
CAT_FEATURES  = ['road_type', 'weather_type']

ALL_EXPECTED = NUM_FEATURES + BOOL_FEATURES + ORD_FEATURES + CAT_FEATURES

missing_from_X    = [c for c in ALL_EXPECTED if c not in X.columns]
unexpected_in_X   = [c for c in X.columns   if c not in ALL_EXPECTED]

print("Schema validation:")
print(f"  Missing from X (will break Pipeline): {missing_from_X if missing_from_X else 'None ✓'}")
print(f"  Extra cols in X (dropped by remainder='drop'): {unexpected_in_X if unexpected_in_X else 'None ✓'}")

Schema validation:
  Missing from X (will break Pipeline): ['is_weekend']
  Extra cols in X (dropped by remainder='drop'): None ✓


In [25]:
# day_of_week: 0=Mon … 6=Sun → is_weekend=1 if Sat(5) or Sun(6)
X = X.copy()
X['is_weekend'] = (X['day_of_week'] >= 5).astype(int)

print(f"is_weekend added. Value counts:\n{X['is_weekend'].value_counts()}")

# Re-run schema check
missing_from_X = [c for c in ALL_EXPECTED if c not in X.columns]
print(f"\nSchema re-check — missing from X: {missing_from_X if missing_from_X else 'None ✓'}")

is_weekend added. Value counts:
is_weekend
0    170342
1     69577
Name: count, dtype: int64

Schema re-check — missing from X: None ✓


In [26]:
null_summary = X[ALL_EXPECTED].isnull().sum()
null_summary = null_summary[null_summary > 0]

if null_summary.empty:
    print("No nulls in any feature column ✓")
else:
    print("Nulls remaining in feature columns:")
    print(null_summary)
    print("\nNote: traffic_level nulls (~34K) are expected — no traffic FK match.")
    print("The Pipeline's OrdinalEncoder will error on NaN. Impute below.")

Nulls remaining in feature columns:
distance_km       1074
traffic_level    32770
road_type        32552
weather_type     24717
dtype: int64

Note: traffic_level nulls (~34K) are expected — no traffic FK match.
The Pipeline's OrdinalEncoder will error on NaN. Impute below.


In [27]:
# traffic_level: fillna with 'medium' (the mode / middle category)
X['traffic_level'] = X['traffic_level'].fillna('medium')

# road_type / weather_type: fillna with mode
for col in ['road_type', 'weather_type']:
    mode_val = X[col].mode(dropna=True)[0]
    before = X[col].isnull().sum()
    X[col] = X[col].fillna(mode_val)
    print(f"{col}: filled {before} nulls with mode='{mode_val}'")

# distance_km: fillna with median (1,074 nulls remain per cleaning doc)
dist_median = X['distance_km'].median()
before = X['distance_km'].isnull().sum()
X['distance_km'] = X['distance_km'].fillna(dist_median)
print(f"distance_km: filled {before} nulls with median={dist_median:.3f}")

# Final null check
remaining = X[ALL_EXPECTED].isnull().sum().sum()
print(f"\nTotal nulls remaining across all feature columns: {remaining}")

road_type: filled 32552 nulls with mode='residential'
weather_type: filled 24717 nulls with mode='clear'
distance_km: filled 1074 nulls with median=4.330

Total nulls remaining across all feature columns: 0


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42    # must never change — keeps train/test rows stable across notebooks
)

# Sanity checks
assert len(set(X_train.index) & set(X_test.index)) == 0, \
    "Index overlap between train and test — train_test_split bug!"

print(f"X_train : {X_train.shape[0]:,} rows × {X_train.shape[1]} cols")
print(f"X_test  : {X_test.shape[0]:,} rows × {X_test.shape[1]} cols")
print(f"y_train : {y_train.shape[0]:,} values")
print(f"y_test  : {y_test.shape[0]:,} values")
print(f"\nTrain/test split: {len(X_train)/len(X)*100:.1f}% / {len(X_test)/len(X)*100:.1f}%")
print("No index overlap between train and test ✓")

X_train : 191,935 rows × 13 cols
X_test  : 47,984 rows × 13 cols
y_train : 191,935 values
y_test  : 47,984 values

Train/test split: 80.0% / 20.0%
No index overlap between train and test ✓


In [29]:
from sklearn.preprocessing import StandardScaler

# ── WRONG: fit scaler on ALL data before split ──────────────────
scaler_leaky = StandardScaler()
X_all_scaled = X[NUM_FEATURES].copy()
X_all_scaled = scaler_leaky.fit_transform(X_all_scaled)   # sees test data → LEAKAGE

print("=== LEAKY approach (fit on full X) ===")
print(f"Scaler mean (distance_km): {scaler_leaky.mean_[0]:.4f}")
print(f"Scaler std  (distance_km): {scaler_leaky.scale_[0]:.4f}")

# ── CORRECT: fit only on X_train, transform X_test ─────────────
scaler_correct = StandardScaler()
scaler_correct.fit(X_train[NUM_FEATURES])                  # never sees test data

X_train_scaled_demo = scaler_correct.transform(X_train[NUM_FEATURES])
X_test_scaled_demo  = scaler_correct.transform(X_test[NUM_FEATURES])  # transform only

print("\n=== CORRECT approach (fit on X_train only) ===")
print(f"Scaler mean (distance_km): {scaler_correct.mean_[0]:.4f}")
print(f"Scaler std  (distance_km): {scaler_correct.scale_[0]:.4f}")

# Compare — means will differ slightly because leaky scaler includes test rows
print(f"\nMean difference (leaky vs correct): "
      f"{abs(scaler_leaky.mean_[0] - scaler_correct.mean_[0]):.6f}")
print("\nConclusion: Inside the sklearn Pipeline, the scaler is fit per fold")
print("on the training slice only — this is the correct approach, automated.")

=== LEAKY approach (fit on full X) ===
Scaler mean (distance_km): 53.3680
Scaler std  (distance_km): 301.2609

=== CORRECT approach (fit on X_train only) ===
Scaler mean (distance_km): 53.7319
Scaler std  (distance_km): 302.6827

Mean difference (leaky vs correct): 0.363909

Conclusion: Inside the sklearn Pipeline, the scaler is fit per fold
on the training slice only — this is the correct approach, automated.


In [30]:
# ── Ordinal encode traffic_level ────────────────────────────────
ord_enc = OrdinalEncoder(categories=[['low', 'medium', 'high']])
X_train_enc = X_train.copy()
X_test_enc  = X_test.copy()

X_train_enc['traffic_level'] = ord_enc.fit_transform(X_train[['traffic_level']])
X_test_enc['traffic_level']  = ord_enc.transform(X_test[['traffic_level']])  # transform only

print("OrdinalEncoder fitted on X_train:")
print(f"  Categories: {ord_enc.categories_}")

# ── One-hot encode road_type and weather_type ───────────────────
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[CAT_FEATURES])

ohe_train = pd.DataFrame(
    ohe.transform(X_train[CAT_FEATURES]),
    columns=ohe.get_feature_names_out(CAT_FEATURES),
    index=X_train.index
)
ohe_test = pd.DataFrame(
    ohe.transform(X_test[CAT_FEATURES]),
    columns=ohe.get_feature_names_out(CAT_FEATURES),
    index=X_test.index
)

X_train_enc = X_train_enc.drop(columns=CAT_FEATURES).join(ohe_train)
X_test_enc  = X_test_enc.drop(columns=CAT_FEATURES).join(ohe_test)

print(f"\nOneHotEncoder fitted on X_train — features out: {ohe.get_feature_names_out(CAT_FEATURES).tolist()}")

# ── StandardScaler on numeric features ─────────────────────────
scaler = StandardScaler()
X_train_enc[NUM_FEATURES] = scaler.fit_transform(X_train_enc[NUM_FEATURES])
X_test_enc[NUM_FEATURES]  = scaler.transform(X_test_enc[NUM_FEATURES])

print(f"\nStandardScaler fitted on X_train. X_train_enc shape: {X_train_enc.shape}")

encoders = {'ordinal': ord_enc, 'onehot': ohe}
print("\nEncoders dict ready: encoders['ordinal'], encoders['onehot']")

OrdinalEncoder fitted on X_train:
  Categories: [array(['low', 'medium', 'high'], dtype=object)]

OneHotEncoder fitted on X_train — features out: ['road_type_residential', 'road_type_urban', 'weather_type_fog', 'weather_type_rain']

StandardScaler fitted on X_train. X_train_enc shape: (191935, 15)

Encoders dict ready: encoders['ordinal'], encoders['onehot']


In [31]:
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"Total rows      : {len(X):,}")
print(f"X_train rows    : {len(X_train):,}")
print(f"X_test rows     : {len(X_test):,}")
print(f"Feature columns : {len(ALL_EXPECTED)}")
print(f"Target          : {'log_travel_time' if USE_LOG_TRANSFORM else 'travel_time_min'}")
print(f"Log transform   : {USE_LOG_TRANSFORM}")
print(f"random_state    : 42")
print()
print("Objects ready for NB-04:")
print("  X_train, X_test  — raw (unscaled, uncoded) — Pipeline handles encoding")
print("  y_train, y_test  — target vector")
print("  USE_LOG_TRANSFORM — flag for predict.py's use_log_transform param")
print()
print("DO NOT pass X_train_enc/X_test_enc into build_pipeline() —")
print("that would double-encode. Pass the raw X_train/X_test instead.")

FEATURE ENGINEERING SUMMARY
Total rows      : 239,919
X_train rows    : 191,935
X_test rows     : 47,984
Feature columns : 13
Target          : log_travel_time
Log transform   : True
random_state    : 42

Objects ready for NB-04:
  X_train, X_test  — raw (unscaled, uncoded) — Pipeline handles encoding
  y_train, y_test  — target vector
  USE_LOG_TRANSFORM — flag for predict.py's use_log_transform param

DO NOT pass X_train_enc/X_test_enc into build_pipeline() —
that would double-encode. Pass the raw X_train/X_test instead.


In [32]:
import os
os.makedirs('../data/processed/splits', exist_ok=True)

X_train.to_csv('../data/processed/splits/X_train.csv', index=True)
X_test.to_csv('../data/processed/splits/X_test.csv',   index=True)
y_train.to_csv('../data/processed/splits/y_train.csv', index=True, header=True)
y_test.to_csv('../data/processed/splits/y_test.csv',   index=True, header=True)

print("Splits saved to ../data/processed/splits/")
print(f"  X_train.csv : {X_train.shape}")
print(f"  X_test.csv  : {X_test.shape}")
print(f"  y_train.csv : {y_train.shape}")
print(f"  y_test.csv  : {y_test.shape}")

Splits saved to ../data/processed/splits/
  X_train.csv : (191935, 13)
  X_test.csv  : (47984, 13)
  y_train.csv : (191935,)
  y_test.csv  : (47984,)
